# NJL3 Phase Diagram — GPU Calculation

Ноутбук-дирижёр: запускает расчёт, визуализирует результаты.
Вся физика — в модулях `gpu/functions/`.

In [7]:
# ── Установка (только при первом запуске в Colab) ──
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !git clone https://github.com/YOUR_USERNAME/NJL3Model_b.git  # замените URL
    %cd NJL3Model_b
    !pip install torch scipy numpy matplotlib -q

In [2]:
import sys, os

# Корень проекта — два уровня выше gpu/notebooks/
# Работает и локально (запуск из gpu/notebooks/), и в Colab (после %cd в корень)
_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)
print('Project root:', _root)

import torch
import numpy as np
import matplotlib.pyplot as plt
import time

import gpu.config as cfg
from gpu.Omega_ren_torch import compute as compute_omega_ren
from gpu.minimizer import find_minimum, save_phase_diagram

print(f'Device: {cfg.DEVICE}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

Project root: /Users/alexeykozhakin/projects/NJL3Model_b
Device: cpu
CUDA available: False


## 1. Параметры расчёта

Правьте здесь перед запуском.

In [3]:
# ── Сетки ──────────────────────────────────────────
N_MU = cfg.N_MU
N_L  = cfg.N_L

mu_grid = torch.linspace(cfg.MU_MIN, cfg.MU_MAX, N_MU, dtype=cfg.DTYPE)
L_grid  = torch.linspace(cfg.L_MIN,  cfg.L_MAX,  N_L,  dtype=cfg.DTYPE)

# ── Регион (b, M) ─────────────────────────────────
region = cfg.REGIONS[0]
b_lo, b_hi, N_B = region['b']
M_lo, M_hi, N_M = region['M']

b_grid = torch.linspace(b_lo, b_hi, N_B, dtype=cfg.DTYPE)
M_grid = torch.linspace(M_lo, M_hi, N_M, dtype=cfg.DTYPE)

print(f'mu  : [{mu_grid[0]:.2f}, {mu_grid[-1]:.2f}]  N={N_MU}')
print(f'L   : [{L_grid[0]:.2f},  {L_grid[-1]:.2f}]   N={N_L}')
print(f'b   : [{b_lo}, {b_hi}]  N={N_B}')
print(f'M   : [{M_lo}, {M_hi}]  N={N_M}')
mem_mb = N_MU * N_L * N_B * N_M * 8 / 1024**2
print(f'Тензор Omega_ren: {N_MU}×{N_L}×{N_B}×{N_M} = {N_MU*N_L*N_B*N_M:,} точек  ({mem_mb:.0f} MB)')

mu  : [3.00, 4.00]  N=50
L   : [1.00,  2.00]   N=50
b   : [0.0, 5.0]  N=100
M   : [0.0, 5.0]  N=100
Тензор Omega_ren: 50×50×100×100 = 25,000,000 точек  (191 MB)


## 2. Расчёт Omega_ren

In [ ]:
t0 = time.time()
omega_ren = compute_omega_ren(
    mu_grid, L_grid, b_grid, M_grid,
    g=cfg.G,
    N_p=cfg.N_P,
    N_phi=cfg.N_P,
    n_modes_max=cfg.N_MODES_MAX,
    chunk_size=cfg.CHUNK_SIZE,
    device=cfg.DEVICE,
    dtype=cfg.DTYPE,
    show_progress=True,          # прогресс-бар по mu
)
print(f'Время расчёта : {time.time()-t0:.1f}s')
print(f'Форма тензора : {tuple(omega_ren.shape)}')
print(f'Диапазон      : [{omega_ren.min():.4f}, {omega_ren.max():.4f}]')

## 3. Поиск минимума

In [ ]:
b_min, M_min, omega_min = find_minimum(omega_ren, b_grid, M_grid, window=5)
print(f'M_min : [{M_min.min():.4f}, {M_min.max():.4f}]')
print(f'b_min : [{b_min.min():.4f}, {b_min.max():.4f}]')

# Предупреждение о точках на границе сетки
n_b = (b_min >= b_grid[-1].item() - 0.01).sum()
n_M = (M_min >= M_grid[-1].item() - 0.01).sum()
if n_b > 0: print(f'⚠️  {n_b} точек на границе b_max — увеличьте b_hi в config.py')
if n_M > 0: print(f'⚠️  {n_M} точек на границе M_max — увеличьте M_hi в config.py')

## 4. Фазовая диаграмма

In [ ]:
mu_np = mu_grid.numpy()
L_np  = L_grid.numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, data, title, cmap in zip(
    axes,
    [M_min.T, b_min.T],
    ['$M_{min}(\\mu, L)$', '$b_{min}(\\mu, L)$'],
    ['viridis', 'plasma'],
):
    im = ax.pcolormesh(mu_np, L_np, data, cmap=cmap, shading='auto')
    plt.colorbar(im, ax=ax)
    ax.set_xlabel('$\\mu$', fontsize=13)
    ax.set_ylabel('$L$', fontsize=13)
    ax.set_title(title, fontsize=14)

plt.tight_layout()
plt.savefig('phase_diagram.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Срез по выбранной точке (mu_i, L_j)

In [ ]:
# ── Выберите точку ─────────────────────────────────
mu_target = 3.0
L_target  = 1.0

i_mu = int(torch.argmin(torch.abs(mu_grid - mu_target)))
i_L  = int(torch.argmin(torch.abs(L_grid  - L_target)))

print(f'mu = {mu_grid[i_mu].item():.3f},  L = {L_grid[i_L].item():.3f}')
print(f'M_min = {M_min[i_mu, i_L]:.4f},  b_min = {b_min[i_mu, i_L]:.4f}')

slab  = omega_ren[i_mu, i_L].cpu().numpy()
b_np2 = b_grid.numpy()
M_np2 = M_grid.numpy()

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Тепловая карта Omega_ren(b, M)
vmax = min(abs(omega_ren[i_mu, i_L].min().item()) * 0.5, 5.0)
im = axes[0].pcolormesh(M_np2, b_np2, slab, cmap='RdBu_r',
                         vmin=-vmax, vmax=vmax, shading='auto')
plt.colorbar(im, ax=axes[0], label='$\\Omega_{ren}$')
axes[0].plot(M_min[i_mu, i_L], b_min[i_mu, i_L], 'k*', ms=14, label='min')
axes[0].set_xlabel('$M$'); axes[0].set_ylabel('$b$')
axes[0].set_title(f'$\\Omega_{{ren}}$  ($\\mu={mu_grid[i_mu]:.2f}$, $L={L_grid[i_L]:.2f}$)')
axes[0].legend()

# Срез по b при M ≈ M_min
i_M_min = int(np.argmin(np.abs(M_np2 - M_min[i_mu, i_L])))
axes[1].plot(b_np2, slab[:, i_M_min])
axes[1].axvline(b_min[i_mu, i_L], color='r', ls='--',
                label=f'b_min={b_min[i_mu,i_L]:.3f}')
axes[1].set_xlabel('$b$'); axes[1].set_ylabel('$\\Omega_{ren}$')
axes[1].set_title(f'Срез по $b$ при $M \\approx {M_np2[i_M_min]:.2f}$')
axes[1].legend()

# Срез по M при b ≈ b_min
i_b_min = int(np.argmin(np.abs(b_np2 - b_min[i_mu, i_L])))
axes[2].plot(M_np2, slab[i_b_min, :])
axes[2].axvline(M_min[i_mu, i_L], color='r', ls='--',
                label=f'M_min={M_min[i_mu,i_L]:.3f}')
axes[2].set_xlabel('$M$'); axes[2].set_ylabel('$\\Omega_{ren}$')
axes[2].set_title(f'Срез по $M$ при $b \\approx {b_np2[i_b_min]:.2f}$')
axes[2].legend()

plt.tight_layout()
plt.show()

## 6. Сохранение результатов

In [ ]:
save_phase_diagram(
    mu_grid, L_grid, b_min, M_min, omega_min,
    output_path=cfg.OUTPUT_JSON,
    meta={'g': cfg.G, 'N_p': cfg.N_P, 'region': str(region)},
)
print('JSON сохранён.')